## naver_news 테이블의 한 행을 표현하기 위한 클래스 생성

In [26]:
# 네이버 뉴스 응답 데이터를 담을 NaverNews 엔티티 클래스 정의
from datetime import datetime

class NaverNews:

    def __init__(self, id: int, title: str, originallink: str, link: str, description: str, pub_date: str, created_at: datetime = None):
        self.__id = id
        self.__title = title
        self.__originallink = originallink
        self.__link = link
        self.__description = description
        self.__pub_date = pub_date
        self.__created_at = created_at

    @property
    def id(self):
        return self.__id

    @property
    def title(self):
        return self.__title

    # __title 속성에 대한 setter 메소드 예시
    # 아래 주석을 해제하면 naver_news.title = "새 제목"처럼 값을 변경할 수 있다.
    # @title.setter
    # def title(self, value):
    #     self.__title = value

    @property
    def originallink(self):
        return self.__originallink

    @property
    def link(self):
        return self.__link

    @property
    def description(self):
        return self.__description

    @property
    def pub_date(self):
        return self.__pub_date

    @property
    def created_at(self):
        return self.__created_at

    # naver new api 응답 데이터를
    # NaverNews 객체로 변환하기 위한 클래스 메서드
    @classmethod
    def from_api_item(cls, item:dict):
        return cls(
            id=None,
            title=item.get('title'),
            originallink=item.get('originallink'),
            link=item.get('link'),
            description=item.get('description'),
            pub_date=item.get('pubDate'),
        )


    def __repr__(self):
        # 객체를 print() 했을 때 어떤 값이 들어 있는지 확인하기 위한 문자열 표현 메소드
        return f'NaverNews({self.__id}, {self.__title}, {self.__originallink}, {self.__link}, {self.__description}, {self.__pub_date}, {self.__created_at})'

## .env를 읽어서 환경변수로 등록

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

NAVER_CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
NAVER_CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

config = {
    "host": os.getenv("DB_HOST", "localhost"), # (key, default)
    "port": os.getenv("DB_PORT", "3306"),  # 기본포트 3306인 경우 생략가능
    "user": os.getenv("DB_USER", "skn_ai"),
    "password": os.getenv("DB_PASSWORD", "1234"),
    "database": os.getenv("DB_DATABASE", "menudb")
}

if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError('NAVER_CLIENT_ID 또는 NAVER_CLIENT_SECRET이 설정되지 않았습니다.')

## 네이버 뉴스 "인공지능" 관련 최근 뉴스 10개 조회해서 DB에 저장

In [27]:
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

# 수집한 news 데이터를 저장할 리스트
naver_news_list: list[NaverNews] = []

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(+url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.json() # 응답 데이터(json) -> dict 변환

    # 응답 데이터 중 뉴스 기사 리스트인 'items'를 NaverNews 객체로 변환해 리스트에 저장
    items = data.get('items', [])
    naver_news_list = [NaverNews.from_api_item(item) for item in items]

    print('response_code: ', response_code)
    print('수집 건수: ', len(naver_news_list))


except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code:  200
수집 건수:  10


In [28]:
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

# 수집한 news 데이터를 저장할 리스트
naver_news_list:list[NaverNews] = []

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(+url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.json() # 응답 데이터(json) -> dict 변환

    # pprint(data)
    # 응답 데이터 중 뉴스 기사 리스트인 'items' 사용
    items = data.get('items', []) # (key, default)

    naver_news_list = [NaverNews.from_api_item(item) for item in items]

    print('response_code: ', response_code)
    print('naver_news_list: ', naver_news_list)


except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code:  200
naver_news_list:  [NaverNews(None, [K생산적금융을 묻다 은행①] 글로벌 자본 모이는 '신뢰의 우산' 만든 싱..., https://www.businesspost.co.kr/BP?command=article_view&num=440006, https://www.businesspost.co.kr/BP?command=article_view&num=440006, 김용진 KB국민은행 싱가포르지점장은 &quot;싱가포르 자체는 첨단산업 생산국이 아니지만 정책적 환경 조성으로 글로벌 생산적금융 투자의 중심지 역할을 하고 있다&quot;며 &quot;<b>인공지능</b>(AI), 데이터센터, 에너지기업들의 해외... , Mon, 15 Jun 2026 17:26:00 +0900, None), NaverNews(None, &quot;혁신기업 고민 해결 지원&quot; 이노션·소뱅 컨설팅 맞손, https://www.mk.co.kr/article/12074607, https://n.news.naver.com/mnews/article/009/0005694146?sid=101, 퀸잇, 런드리고, 라엘, 그래비티랩스 등이 참여한다. 참여 기업이 직면한 시장 확장, <b>인공지능</b>(AI)·데이터 활용, 브랜드 성장, 신규 수익 모델 구축 등 핵심 과제를 중심으로 실질적 협업 방안을 도출할 계획이다., Mon, 15 Jun 2026 17:26:00 +0900, None), NaverNews(None, &quot;일본 ESS 시장 돈 된다&quot; LS일렉·효성重 잰걸음, https://www.mk.co.kr/article/12074605, https://n.news.naver.com/mnews/article/009/0005694144?sid=101, 재생에너지 매년 35% 급성장 시공·운영으로 사업확장 나서 국내 전력기기업체가 <b>인공지능</b>(AI)발 전력 수요가 증가하고 재생에너지 발전이 확대되고 있는 일본 에너

In [17]:
import mysql.connector

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                insert into naver_news
                    (title, originallink, link, description, pub_date)
                    VALUES (%s, %s, %s, %s, %s)
                ''', (naver_news.title,
                       naver_news.originallink,
                       naver_news.link,
                       naver_news.description,
                       naver_news.pub_date
                   ))
            conn.commit() # 전체 insert 내용 커밋



except mysql.connector.Error as err:
    print("DB에러: ", err)
    conn.rollback() # 오류 발생 시 전체 rollback

In [18]:
# DB에 저장된 네이버 뉴스 데이터를 조회해 객체 리스트로 변환
try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            cursor.execute('''
                           select *
                           from naver_news
                           order by id desc
                           ''')  # sql, params
            naver_news_list = [NaverNews(*row) for row in cursor.fetchall()]
            print(naver_news_list)
except mysql.connector.Error as e:
    print('DB 오류:', e)

[NaverNews(10, 김한종 장성군수, 군정 복귀 첫 업무로 황룡면 AI데이터센터 점검, https://www.newstnt.com/news/articleView.html?idxno=706394, https://www.newstnt.com/news/articleView.html?idxno=706394, 성과를 거둘 수 있도록 사업 추진에 한 치의 차질도 없도록 최선을 다해 달라&quot;고 말했다. 장성군은 황룡면 AI데이터센터 건립을 통해 <b>인공지능</b> 산업 기반을 조성하고 지역경제 활성화와 일자리 창출 효과를 기대하고 있다., Mon, 15 Jun 2026 16:34:00 +0900, 2026-06-15 16:49:53), NaverNews(9, 우리銀, 효성그룹 전력·첨단소재 투자에 2조 지원, https://www.sedaily.com/article/20055940?ref=naver, https://n.news.naver.com/mnews/article/011/0004631298?sid=101, 데이터센터와 <b>인공지능</b>(AI) 확산에 따른 미국 내 전력기기 수요 증가에 대응하고 공급망 우위를 확보하기 위해 생산 역량 강화에 집중하고 있다. 효성티앤씨 역시 첨단 소재 분야에서 경쟁력을 확보하고 있다. 회사는 세계... , Mon, 15 Jun 2026 16:34:00 +0900, 2026-06-15 16:49:53), NaverNews(8, MS 코파일럿코워크, 구글·슬랙과 다른 점은 'M365 업무 맥락', https://www.bloter.net/news/articleView.html?idxno=665462, https://n.news.naver.com/mnews/article/293/0000086285?sid=105, 마이크로소프트(MS)는 <b>인공지능</b>(AI) 에이전트 기반 협업 도구 '코파일럿 코워크(Copilot Cowork)'의 강점으로 마이크로소프트 365(M365) 기반의 업무 맥락 이해를 내세웠다. 코파일럿 

## 중복 뉴스 방지
- 중복 뉴스(중복 행)를 방지하기 위해서는
  중복 기준이 되는 컬럼을 설정하고,
  컬럼 값이 중복이면 update가 수행되게 해야함.

- 중복 판단을 하려면 PK 또는 UNIQUE 제약조건을 이용
    - 빠른 중복 판단 가능

- `replace` 대신 `on duplicate key update` 구문 사용
    - replace : delete -> insert
    - on duplicate key update : update 1회

```sql
alter table naver_news
add constraint uq_naver_news_link unique (link);
```

In [29]:
insert_count = 0
skip_count = 0

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                    insert into naver_news
                        (title, originallink, link, description, pub_date)
                    values
                        (%s, %s, %s, %s, %s)
                    on duplicate key update
                        link = link
                ''', (
                    naver_news.title,
                    naver_news.originallink,
                    naver_news.link,
                    naver_news.description,
                    naver_news.pub_date
                ))

                if cursor.rowcount == 1:
                    insert_count += 1
                else:
                    skip_count += 1

            conn.commit()

    print(f"신규 저장: {insert_count}건")
    print(f"중복 제외: {skip_count}건")

except mysql.connector.Error as e:
    print('DB 오류:', e)

신규 저장: 10건
중복 제외: 0건
